# Pediatrician Scheduling at British Columbia Women's Hospital
## Optimization Model Formulation

---
## 1. Sets and Indices

**Primary indices:**

- Let $i$ represent the number of shifts in the 3-week cycle, where $i = 1, 2, \ldots, 42$
- Let $j$ represent the index for pediatricians (peds), where $j = 1, 2, \ldots, 13$
- Let $k$ represent the index for weeks in the cycle, where $k = 1, 2, 3$

**Shift numbering scheme:**

Each week contains 14 shifts (day and night for each of the 7 days). The shift index within a week follows the pattern: Monday Day, Monday Night, Tuesday Day, Tuesday Night, …, Sunday Day, Sunday Night. Weeks 2 and 3 continue sequentially (shifts 15–28 and 29–42, respectively).

| Week | Day | Shift Type | Shift # |
|------|-----|------------|---------|
| 1 | Mon | Day / Night | 1 / 2 |
| 1 | Tue | Day / Night | 3 / 4 |
| 1 | Wed | Day / Night | 5 / 6 |
| 1 | Thu | Day / Night | 7 / 8 |
| 1 | Fri | Day / Night | 9 / 10 |
| 1 | Sat | Day / Night | 11 / 12 |
| 1 | Sun | Day / Night | 13 / 14 |
| 2 | Mon–Sun | Day / Night | 15–28 |
| 3 | Mon–Sun | Day / Night | 29–42 |

**Key subsets:**

- Let $D = \{1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 41\}$ denote the set of **day shifts**
- Let $N = \{2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42\}$ denote the set of **night shifts**
- Let $N_k$ denote the set of night shifts in week $k$:
  - $N_1 = \{2, 4, 6, 8, 10, 12, 14\}$
  - $N_2 = \{16, 18, 20, 22, 24, 26, 28\}$
  - $N_3 = \{30, 32, 34, 36, 38, 40, 42\}$
- Let $W_k$ denote the set of **weekend shifts** (Friday Night, Saturday Day & Night, Sunday Day & Night) in week $k$:
  - $W_1 = \{10, 11, 12, 13, 14\}$
  - $W_2 = \{24, 25, 26, 27, 28\}$
  - $W_3 = \{38, 39, 40, 41, 42\}$
- Let $POW_k$ denote the set of shifts the **Pediatrician of the Week (POW)** must cover in week $k$ (Tuesday Day, Thursday Day, Saturday Night):
  - $POW_1 = \{3, 7, 12\}$
  - $POW_2 = \{17, 21, 26\}$
  - $POW_3 = \{31, 35, 40\}$
- Let $CN$ denote the set of **consecutive night shift pairs** $(i, i')$ where shifts $i$ and $i'$ fall on adjacent calendar days. Within each week these are 6 adjacent-night pairs; cross-week pairs are $(14, 16)$ and $(28, 30)$.
- Let $T$ denote the set of **night shift pairs two calendar days apart** $(i, i')$: e.g., Monday Night & Wednesday Night within a week, or Saturday Night & Monday Night across consecutive weeks.

---
## 2. Parameters

| Symbol | Description |
|--------|-------------|
| $s_j$ | Number of shifts pediatrician $j$ is required to work in the 3-week cycle (from *ShiftsRequired* tab) |
| $a_{ij}$ | $a_{ij} = 1$ if pediatrician $j$ has **not** requested shift $i$ off (i.e., is available), $0$ otherwise (from *ShiftRequests* tab) |
| $p_j$ | Number of remaining POW assignments required for pediatrician $j$ for the year; if $p_j = 0$ the ped cannot be assigned as POW (from *RemainingPOWrequirements* tab) |
| $w_g$ | Weight assigned to goal $g = 1, 2, 3, 4$ in the objective function |


---
## 3. Decision Variables

**Primary binary variables:**

- Let $x_{ij} = 1$ if pediatrician $j$ works shift $i$, $0$ otherwise $\quad \forall\, i = 1, \ldots, 42;\; j = 1, \ldots, 13$
- Let $y_{jk} = 1$ if pediatrician $j$ is designated as the Pediatrician of the Week for week $k$, $0$ otherwise $\quad \forall\, j = 1, \ldots, 13;\; k = 1, 2, 3$

**Auxiliary binary variable (for goal programming):**

- Let $\delta_{jk} = 1$ if pediatrician $j$ works at least one shift in the weekend of week $k$, $0$ otherwise

**Deviation variables — non-negative (for goal programming):**

| Variable | Meaning |
|----------|---------|
| $cw_{jk} \geq 0$ | Captures whether ped $j$ works in **both** consecutive weekends $k$ and $k+1$ (for $k = 1, 2$) |
| $e2_{j,i,i'} \geq 0$ | Captures violation of the "no night shifts two days apart" preference for ped $j$ and pair $(i, i') \in T$ |
| $d_j^+,\; d_j^- \geq 0$ | Over- and under-deviation from an equal day/night split for ped $j$ |

---
## 4. Hard Constraints

The following constraints reflect hospital policy and are **non-negotiable**.

---

### C1: Assign Exactly 1 Pediatrician to Each Shift

Every shift must be covered by exactly one pediatrician:

$$\sum_{j=1}^{13} x_{ij} = 1 \qquad \forall\, i = 1, 2, \ldots, 42$$

---

### C2: Each Pediatrician Works the Required Number of Shifts

Each ped's total shifts across the cycle must equal their required workload $s_j$:

$$\sum_{i=1}^{42} x_{ij} = s_j \qquad \forall\, j = 1, 2, \ldots, 13$$

---

### C3: Respect Shift Requests

A pediatrician may only be assigned to shifts they have not requested off:

$$x_{ij} \leq a_{ij} \qquad \forall\, i = 1, \ldots, 42;\; j = 1, \ldots, 13$$

---

### C4: No Back-to-Back Shifts (No 24 Consecutive Hours)

No pediatrician may work shift $i$ immediately followed by shift $i+1$:

$$x_{ij} + x_{i+1,j} \leq 1 \qquad \forall\, i = 1, \ldots, 41;\; j = 1, \ldots, 13$$

---

### C5: No Consecutive Night Shifts

No pediatrician may work night shifts on two adjacent calendar days. Let $CN$ denote the set of all consecutive-night pairs:

$$x_{ij} + x_{i'j} \leq 1 \qquad \forall\, (i, i') \in CN;\; j = 1, \ldots, 13$$

---

### C6: At Most 2 Shifts per Weekend

A weekend consists of Friday Night, Saturday Day & Night, and Sunday Day & Night ($|W_k| = 5$ shifts). No ped may be scheduled for more than 2 weekend shifts in a given weekend:

$$\sum_{i \in W_k} x_{ij} \leq 2 \qquad \forall\, k = 1, 2, 3;\; j = 1, \ldots, 13$$

---

### C7: Exactly One POW per Week

$$\sum_{j=1}^{13} y_{jk} = 1 \qquad \forall\, k = 1, 2, 3$$

---

### C8: POW Must Cover Required Shifts

The POW for week $k$ must work the Tuesday Day, Thursday Day, and Saturday Night shifts (set $POW_k$):

$$x_{ij} \geq y_{jk} \qquad \forall\, i \in POW_k;\; j = 1, \ldots, 13;\; k = 1, 2, 3$$

---

### C9: At Most One POW Assignment per Cycle

No pediatrician can serve as POW more than once in the 3-week cycle:

$$\sum_{k=1}^{3} y_{jk} \leq 1 \qquad \forall\, j = 1, \ldots, 13$$

---

### C10: Fulfilled Annual POW Requirements

Peds whose annual POW requirement is already met ($p_j = 0$; i.e., peds 3, 5, and 13) cannot be assigned as POW:

$$\sum_{k=1}^{3} y_{jk} = 0 \qquad \forall\, j \text{ where } p_j = 0$$

---

### C11: Binary and Non-Negativity Requirements

$$x_{ij} \in \{0, 1\} \qquad \forall\, i = 1, \ldots, 42;\; j = 1, \ldots, 13$$

$$y_{jk} \in \{0, 1\} \qquad \forall\, j = 1, \ldots, 13;\; k = 1, 2, 3$$

---
## 5. Soft Constraints and Goal Programming

The following constraints reflect the *preferences* of the pediatricians. Violating them does not breach hospital policy, but doing so too often reduces schedule quality. Because there are **four competing preferences** to satisfy simultaneously, we apply **goal programming**: we introduce deviation variables that measure how far the schedule falls from each goal, then minimize a weighted combination of those deviations.

---

### G1: Prefer Not to Work Consecutive Weekends

Define auxiliary binary $\delta_{jk} = 1$ if ped $j$ works at least one shift in weekend $k$, enforced by:

$$\delta_{jk} \leq \sum_{i \in W_k} x_{ij} \qquad \forall\, j,\, k$$

$$\sum_{i \in W_k} x_{ij} \leq 5 \cdot \delta_{jk} \qquad \forall\, j,\, k \quad (\text{since } |W_k| = 5)$$

A violation occurs when ped $j$ works in **both** consecutive weekends $k$ and $k+1$. The deviation variable $cw_{jk}$ captures this:

$$cw_{jk} \geq \delta_{jk} + \delta_{j,k+1} - 1 \qquad \forall\, k = 1, 2;\; j = 1, \ldots, 13$$

$$cw_{jk} \geq 0$$

> $cw_{jk} > 0$ indicates a consecutive-weekend preference violation for ped $j$ across weekends $k$ and $k+1$.

---

### G2: Prefer Not to Work Night Shifts Two Days Apart

For each pair of night shifts $(i, i') \in T$ and each ped $j$, the soft goal is $x_{ij} + x_{i'j} \leq 1$. The deviation variable $e2_{j,i,i'}$ absorbs any violation:

$$x_{ij} + x_{i'j} \leq 1 + e2_{j,i,i'} \qquad \forall\, (i, i') \in T;\; j = 1, \ldots, 13$$

$$e2_{j,i,i'} \geq 0$$

---

### G3: Prefer No More Than Two Night Shifts per Work Week

For each ped $j$ and week $k$, the ideal is $\leq 2$ night shifts. The deviation variable $e3_{jk}$ captures the excess:

$$\sum_{i \in N_k} x_{ij} \leq 2 + e3_{jk} \qquad \forall\, k = 1, 2, 3;\; j = 1, \ldots, 13$$

$$e3_{jk} \geq 0$$

---

### G4: Equitable Day/Night Split

Each ped's total shift load should be evenly split between day and night. The target number of day shifts for ped $j$ is $\lfloor s_j / 2 \rfloor$. The deviation variables $d_j^+$ and $d_j^-$ track over- and under-assignment of day shifts:

$$\sum_{i \in D} x_{ij} + d_j^- - d_j^+ = \left\lfloor \frac{s_j}{2} \right\rfloor \qquad \forall\, j = 1, \ldots, 13$$

$$d_j^+,\; d_j^- \geq 0$$

> $d_j^+ > 0$ means more day shifts than the target; $d_j^- > 0$ means fewer day shifts than the target.

---
## 6. Objective Function

The objective is to **minimise** a weighted sum of all goal deviations across the four preferences:

$$\min \; Z = w_1 \sum_{j} \sum_{k=1}^{2} cw_{jk} \;+\; w_2 \sum_{j} \sum_{(i,i') \in T} e2_{j,i,i'} \;+\; w_3 \sum_{j} \sum_{k=1}^{3} e3_{jk} \;+\; w_4 \sum_{j=1}^{13} \left( d_j^+ + d_j^- \right)$$

where $w_1, w_2, w_3, w_4 \geq 0$ are the weights for Goals 1–4, respectively.

| Term | Deviation variable(s) | Goal | Minimising achieves… |
|------|-----------------------|------|----------------------|
| $w_1$ | $cw_{jk}$ | G1: No consecutive weekends | Fewer peds working back-to-back weekends |
| $w_2$ | $e2_{j,i,i'}$ | G2: No nights two days apart | Fewer near-consecutive night pairings |
| $w_3$ | $e3_{jk}$ | G3: ≤2 night shifts per week | Fewer weeks with 3+ nights for one ped |
| $w_4$ | $d_j^+, d_j^-$ | G4: Equitable day/night split | Workload balanced across day and night |

> If all weights are equal, all four goals receive the same priority. These weights can be tuned to reflect Meg's scheduling priorities.

---
## 7. Complete Model Summary

$$\min \; Z = w_1 \sum_{j} \sum_{k=1}^{2} cw_{jk} + w_2 \sum_{j} \sum_{(i,i') \in T} e2_{j,i,i'} + w_3 \sum_{j} \sum_{k=1}^{3} e3_{jk} + w_4 \sum_{j=1}^{13} \left( d_j^+ + d_j^- \right)$$

**Subject to:**

| # | Constraint | Type |
|---|-----------|------|
| C1 | $\sum_{j=1}^{13} x_{ij} = 1 \quad \forall\, i$ | Hard |
| C2 | $\sum_{i=1}^{42} x_{ij} = s_j \quad \forall\, j$ | Hard |
| C3 | $x_{ij} \leq a_{ij} \quad \forall\, i, j$ | Hard |
| C4 | $x_{ij} + x_{i+1,j} \leq 1 \quad \forall\, i, j$ | Hard |
| C5 | $x_{ij} + x_{i'j} \leq 1 \quad \forall\, (i,i') \in CN, j$ | Hard |
| C6 | $\sum_{i \in W_k} x_{ij} \leq 2 \quad \forall\, k, j$ | Hard |
| C7 | $\sum_{j=1}^{13} y_{jk} = 1 \quad \forall\, k$ | Hard |
| C8 | $x_{ij} \geq y_{jk} \quad \forall\, i \in POW_k, j, k$ | Hard |
| C9 | $\sum_{k=1}^{3} y_{jk} \leq 1 \quad \forall\, j$ | Hard |
| C10 | $\sum_{k=1}^{3} y_{jk} = 0 \quad \forall\, j: p_j = 0$ | Hard |
| C11 | $x_{ij} \in \{0,1\},\; y_{jk} \in \{0,1\}$ | Hard |
| G1 | $cw_{jk} \geq \delta_{jk} + \delta_{j,k+1} - 1 \quad \forall\, k=1,2, j$ | Soft |
| G2 | $x_{ij} + x_{i'j} \leq 1 + e2_{j,i,i'} \quad \forall\, (i,i') \in T, j$ | Soft |
| G3 | $\sum_{i \in N_k} x_{ij} \leq 2 + e3_{jk} \quad \forall\, k, j$ | Soft |
| G4 | $\sum_{i \in D} x_{ij} + d_j^- - d_j^+ = \lfloor s_j/2 \rfloor \quad \forall\, j$ | Soft |

**Model type:** Mixed Integer Linear Program (MILP)

**Scale:**
- $13 \times 42 = 546$ primary binary variables ($x_{ij}$)
- $13 \times 3 = 39$ POW binary variables ($y_{jk}$)
- Auxiliary deviation variables for four goals

**Recommended solvers:** Gurobi, CPLEX, or open-source CBC